<!-- 
Data Set Source

Nath, Prious Chandra; Zahra, Farhin; Islam, Aminul; Shah, MD. Meraz Ahasan; Miah, M. Saef Ullah; Rahman, M. Mostafizur (2026), “BDFD9: An annotated image dataset of nine fabric defect types collected in Bangladeshi textile and garment factories”, Mendeley Data, V3, doi: 10.17632/z49cc7zvfc.3

https://data.mendeley.com/datasets/z49cc7zvfc/3 

-->

In [ ]:
from pathlib import Path
import pandas as pd

DATASET_DIR = Path("dataset")

# Read classes
print("CLASSES.TXT")
print((DATASET_DIR / "classes.txt").read_text())

# Read YAML
print("\nDATA.YAML")
print((DATASET_DIR / "data.yaml").read_text())

# Read metadata

metadata = pd.read_csv(DATASET_DIR / "dataset_metadata.csv")
display(metadata.head())

print(metadata.shape)
print(metadata.columns.tolist())
print(metadata.dtypes)

In [2]:
# Exploratory data analysis (EDA)

# Check number of images in each split
metadata['split'].value_counts()

split
train    775
valid    208
test     105
Name: count, dtype: int64

In [3]:
# Check class distribution
metadata['classes'].value_counts()

classes
Yurn Dust                 192
Vertical LineMark         179
Slub                      118
Knot                      113
Setup                     105
Horizontal Mark            93
Oil Spot                   86
Hole                       80
DropStitch                 79
DropStitch;Hole            37
Hole;Setup                  3
Unlabeled_NoAnnotation      2
Hole;Yurn Dust              1
Name: count, dtype: int64

In [4]:
#Check class distribution across splits
pd.crosstab(metadata['classes'], metadata['split'])

split,test,train,valid
classes,,,
DropStitch,8,54,17
DropStitch;Hole,4,27,6
Hole,8,56,16
Hole;Setup,0,2,1
Hole;Yurn Dust,0,1,0
Horizontal Mark,8,66,19
Knot,12,77,24
Oil Spot,4,74,8
Setup,11,74,20


In [5]:
#Check the number of defect instances
metadata['num_boxes'].describe()

count    1088.000000
mean        1.275735
std         1.307849
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        38.000000
Name: num_boxes, dtype: float64

In [6]:
#Check whether images contain multiple classes
metadata['classes'].nunique(), metadata['classes'].isna().sum()

(13, np.int64(0))

### SUMMARY
 1045 images have only one defect class
 41 images have two defect classes
 2 images have no annotation

For this task of classification for now, my target is to classify images with only one label, so we'll work with 1045 single labeled classes

Also, the largest class has 192 images, while the smallest has 79. A ratio of 2.43:1

Moderately class imbalance

Also, with the current split, for example, oil spot has 74 training examples but only 8 validation images, relatively few images for validation.

describe

Mean: On average, each image has approximately 1.28 annotated boxes.

Median: Half of your images have one or fewer boxes, and half have one or more.

Standard deviation: Measures how spread out the values are around the mean. Here, the relatively high value compared with the mean suggests that some images have substantially more boxes than others.

Quartiles: The 25th, 50th, and 75th percentiles divide sorted observations into four sections.

Notice that all three quartiles equal 1. This means at least 75% of the images have one or fewer boxes.

13 unique class strings. 


In [7]:
#Inspect the multi-label images and their box counts:

multi_label = metadata[
    metadata['classes'].str.contains(';', na=False)
]

display(multi_label)

,filename,label_file,split,classes,num_boxes,class_folders
178,IMG_1082_webp.rf.f6TnbKr49WkhdFXv3E6P.webp,IMG_1082_webp.rf.f6TnbKr49WkhdFXv3E6P.txt,train,DropStitch;Hole,2,DropStitch;Hole
184,IMG_1083_webp.rf.Z6pDbaoRTNz91rUbJLKT.webp,IMG_1083_webp.rf.Z6pDbaoRTNz91rUbJLKT.txt,train,DropStitch;Hole,2,DropStitch;Hole
190,IMG_1085_webp.rf.rKhqzBp3AKbohqcLsPjG.webp,IMG_1085_webp.rf.rKhqzBp3AKbohqcLsPjG.txt,train,DropStitch;Hole,2,DropStitch;Hole
201,IMG_1088_webp.rf.Su20EwLr2CazezonWWsk.webp,IMG_1088_webp.rf.Su20EwLr2CazezonWWsk.txt,train,DropStitch;Hole,2,DropStitch;Hole
214,IMG_1092_webp.rf.lZ5UiDtn0iFWOjpjjfb3.webp,IMG_1092_webp.rf.lZ5UiDtn0iFWOjpjjfb3.txt,train,DropStitch;Hole,2,DropStitch;Hole
231,IMG_1098_webp.rf.ejwsoZdnuJkWAynBYU3b.webp,IMG_1098_webp.rf.ejwsoZdnuJkWAynBYU3b.txt,train,DropStitch;Hole,2,DropStitch;Hole
241,IMG_1102_webp.rf.YSOcmqPXDVSlJ5tEihvv.webp,IMG_1102_webp.rf.YSOcmqPXDVSlJ5tEihvv.txt,train,DropStitch;Hole,2,DropStitch;Hole
271,IMG_1116_webp.rf.FvSpBcJNF3Dr0gsD6Xlw.webp,IMG_1116_webp.rf.FvSpBcJNF3Dr0gsD6Xlw.txt,train,DropStitch;Hole,2,DropStitch;Hole
276,IMG_1117_webp.rf.vs4Q9pUF8KW1wVngfjLK.webp,IMG_1117_webp.rf.vs4Q9pUF8KW1wVngfjLK.txt,train,DropStitch;Hole,2,DropStitch;Hole
282,IMG_1120_webp.rf.NqDGw5x0FEhqMSDezueg.webp,IMG_1120_webp.rf.NqDGw5x0FEhqMSDezueg.txt,train,DropStitch;Hole,2,DropStitch;Hole


In [8]:
#Check the split distribution using only single-class images:

single_label = metadata[
    ~metadata['classes'].str.contains(';', na=False)
    & (metadata['classes'] != 'Unlabeled_NoAnnotation')
]

pd.crosstab(
    single_label['classes'],
    single_label['split']
)

split,test,train,valid
classes,,,
DropStitch,8,54,17
Hole,8,56,16
Horizontal Mark,8,66,19
Knot,12,77,24
Oil Spot,4,74,8
Setup,11,74,20
Slub,12,83,23
Vertical LineMark,18,125,36
Yurn Dust,19,135,38


In [9]:
single_label['split'].value_counts()


split
train    744
valid    201
test     100
Name: count, dtype: int64